In [2]:
import pandas as pd
import numpy as np

In [ ]:
dfG = pd.read_excel('TimeSeries_FULL_GMonth.xlsx')
dfU = pd.read_excel('TimeSeries_FULL_UMonth.xlsx')


In [18]:
General = dfG.copy()
# General

In [19]:
masterF = 'Real_Master_CPI.xlsx'
Real_master = pd.read_excel(masterF)
# Merge Real_master (RM)
RM = Real_master[['รหัส','ชุด', 'รายการ','กำหนดให้เก็บ','รายละเอียด']].rename(columns={'รหัส': 'CODE7'})
RM["CODE7"] = RM["CODE7"].astype(str).str.zfill(7)
# --- เพิ่มบรรทัดนี้เพื่อแก้ปัญหา TypeError ---
RM['กำหนดให้เก็บ'] = pd.to_numeric(RM['กำหนดให้เก็บ'], errors='coerce').fillna(0)

# ตอนนี้จะสามารถ Run บรรทัดนี้ได้แล้วครับ
RM = RM[(RM['ชุด']!='ชุด อ.') & (RM['กำหนดให้เก็บ'] > 0)]
RM = RM[['CODE7', 'รายการ']]
RM['CODE7'] = RM['CODE7'].astype(str)

RM

,CODE7,รายการ
5,1112104,แป้งทอดกรอบ
10,1112205,ขนมปังปอนด์
12,1112207,อาหารธัญพืช
15,1112210,ขนมอบ
80,1132001,นมสด
...,...,...
701,6311002,ค่าอาหารสำหรับตักบาตร
702,7100001,บุหรี่
704,7210001,เบียร์
705,7220001,ไวน์


In [20]:
General.head()
# ds

,TYPE,GROUP,DESCRIPTION,SHOP_NAME,ประเภทร้านค้า,CODE_7_DIGITS,COMMODITY_CODE,DILLER_CODE,ComKeyBasic,AVG6601,...,Relstreak_max_nonnull,Relstreak_max_eq100,Reltrail_nonnull,Reltrail_eq100,AVG_maxdiff,AVG_mindiff,CountAVG,AVG_Max,AVG_Min,AVGtail_stable
0,10,กทม.กลุ่มจตุจักร,กระโปรงสตรี ผ้าฝ้ายผสมโพลีเอสเตอร์ มีซับใน ทร...,ห้างเซ็นทรัล ลาดพร้าว,Central,2124003,2124003111012319,103000157,21240031110123190103000157,NaN,...,11,4,11,3,86.0,-365.00,11,1795.0,1350.00,4.0
1,10,กทม.กลุ่มจตุจักร,นิตยสารรายเดือน/ข่าว /National Geographic,นายอินทร์ สาขาห้างบิ๊กซีเอ็กตราลาดพร้าว,Big C,6140004,6140004111010105,103000363,61400041110101050103000363,69.0,...,24,24,0,0,0.0,0.00,24,69.0,69.00,NaN
2,10,กทม.กลุ่มจตุจักร,-แผ่นยิบซั่ม ฝ้าเพดาน ชนิดธรรมดา ขอบลาด ขนาดกว...,ร้านถนัดค้าวัสดุก่อสร้าง,ร้านอื่นๆ,3130009,3130009111011680,103000376,31300091110116800103000376,NaN,...,21,21,0,0,0.0,0.00,21,200.0,200.00,NaN
3,10,กทม.กลุ่มจตุจักร,-โฟมล้างหน้า แบบครีม สำหรับผู้ชาย บรรจุหลอดพลา...,ห้างบิ๊กซีเอ็กตร้า สาขาลาดพร้าว2,Big C,4210022,4210022111010307,109900046,42100221110103070109900046,129.0,...,17,3,0,0,40.0,-30.00,27,139.0,89.00,NaN
4,10,กทม.กลุ่มจตุจักร,กระดาษชำระ ชนิดม้วน 2 ชั้น หยาบ บรรจุห่...,ห้างบิ๊กซีเอ็กตร้า สาขาลาดพร้าว2,Big C,4210011,4210011121010402,109900046,42100111210104020109900046,NaN,...,20,4,20,4,9.0,-9.75,20,44.0,29.25,5.0


In [21]:
General = dfG.copy()
General = General.rename(columns={'GROUP':'PROVINCE_NAME'})
General["DILLER_CODE"] = General["DILLER_CODE"].astype(str).str.zfill(10)
# ===================
import pandas as pd
import glob
# ค้นหาไฟล์ทั้งหมดที่ขึ้นต้นด้วย shop_data และลงท้ายด้วย .xlsx
region_map = {
    'กลาง': ["CU", "CC"], 'เหนือ': ["NU", "NN"], 
    'ใต้': ["SU", "SS"], 'ตะวันออกเฉียงเหนือ': ["EU", "EE"]
}
General['ภาค'] = None
for region, codes in region_map.items():
    General.loc[General['TYPE'].isin(codes), 'ภาค'] = region

target_provinces = ['นนทบุรี', 'สมุทรปราการ', 'ปทุมธานี']
General.loc[General['PROVINCE_NAME'].isin(target_provinces), 'ภาค'] = 'ปริมณฑล'
General['ภาค'] = General['ภาค'].fillna('กทม.')
General['PROVINCE_NAME'] = General['PROVINCE_NAME'].replace('กรุงเทพมหานคร', 'กทม.')


files_shop = glob.glob(str("shop_data*.xlsx"))
files_shop
ds_ = pd.read_excel(files_shop [0], skiprows=2, engine='calamine')
ds = ds_[["รหัสร้าน", "กลุ่ม"]].rename(columns={"รหัสร้าน": "DILLER_CODE"})
ds["DILLER_CODE"] = ds["DILLER_CODE"].astype(str).str.zfill(10)
ds["กลุ่ม"] = ds["กลุ่ม"].replace(["-", " "], "", regex=True)
General = General.merge(ds, on=["DILLER_CODE"], how="left")
# สร้าง Column GROUP
# General['GROUP'] = General['PROVINCE_NAME'].fillna('').astype(str) + df['กลุ่ม'].fillna('').astype(str)
General = General.loc[General['กลุ่ม'] == 'กลุ่มเตาปูน'].copy() 
General = General[General["AVG6901"]>0]
# General = General[General['GROUP'] == 'อุตรดิตถ์']
# ===================


General = General.rename(columns={"CODE_7_DIGITS":"CODE7"})
General['COMMODITY_CODE'] = General['COMMODITY_CODE'].astype(str).str.zfill(16)
General['DILLER_CODE'] = General['DILLER_CODE'].astype(str).str.zfill(10)
General["CODE7"] = General["CODE7"].astype(str).str.zfill(7)
General = pd.merge(RM,General,on='CODE7',how = 'left')
General = General.sort_values(by='CODE7').reset_index(drop=True)

General = General.rename(columns={'AVGtail_stable': 'ราคาคงที่กี่เดือน'})

# 3. ลบคอลัมน์ที่ไม่ต้องการออก (Drop)
cols_to_drop = [
    'REL6601', 'REL6602', 'REL6603', 'REL6604', 'REL6605', 'REL6606', 
    'REL6607', 'REL6608', 'REL6609', 'REL6610', 'REL6611', 'REL6612', 
    'REL6701', 'REL6702', 'REL6703', 'REL6704', 'REL6705', 'REL6706', 
    'REL6707', 'REL6708', 'REL6709', 'REL6710', 'REL6711', 'REL6712', 
    'REL6801', 'REL6802', 'REL6803', 'REL6804', 'REL6805', 'REL6806', 
    'REL6807', 'REL6808', 'REL6809', 'REL6810', 'REL6811', 'REL6812', 
    'REL6901', 'Rel_Max', 'Rel_Min', 'Relstreak_max_nonnull', 
    'Relstreak_max_eq100', 'Reltrail_nonnull', 'Reltrail_eq100', 
    'AVG_maxdiff', 'AVG_mindiff', 'CountAVG', 'AVG_Max', 'AVG_Min'
]

# ใช้ errors='ignore' เพื่อป้องกันกรณีบางคอลัมน์ไม่มีอยู่ใน DataFrame จะได้ไม่ error
General = General.drop(columns=cols_to_drop, errors='ignore')

output_file = "Steak_Month_ชุดทั่วไป.xlsx"
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    General.to_excel(writer, index=False, sheet_name='Data')
    
    workbook  = writer.book
    worksheet = writer.sheets['Data']
    
    # Formats
    header_fmt = workbook.add_format({'bold':True, 'bg_color':'#D7E4BC', 'border':1, 'align':'center'})
    line_fmt   = workbook.add_format({'bottom': 2}) # เส้นหนาแบ่งกลุ่ม
    num_fmt    = workbook.add_format({'num_format': '#,##0.00'})

    # 1. เขียน Header ใหม่ด้วย Format
    for col_num, value in enumerate(General.columns):
        worksheet.write(0, col_num, value, header_fmt)
    
    # 2. Freeze Panes: แถวที่ 1 (Header), คอลัมน์ที่ 0
    worksheet.freeze_panes(1, 10)
    
    # 3. วาดเส้นแบ่งกลุ่มตาม CODE7
    code7_idx = General.columns.get_loc('CODE7')
    for row in range(1, len(General)):
        if General.iloc[row-1, code7_idx] != General.iloc[row, code7_idx]:
            worksheet.set_row(row, None, line_fmt) # ขีดเส้นใต้แถวก่อนหน้า
            
    # ปรับความกว้างคอลัมน์พื้นฐาน
    worksheet.set_column(0, len(General.columns)-1, 12)

print(f"เสร็จสมบูรณ์! ไฟล์ถูกบันทึกที่: {output_file}")




เสร็จสมบูรณ์! ไฟล์ถูกบันทึกที่: Steak_Month_ชุดทั่วไป.xlsx


In [9]:
General.head()

,TYPE,GROUP,DESCRIPTION,SHOP_NAME,ประเภทร้านค้า,CODE_7_DIGITS,COMMODITY_CODE,DILLER_CODE,ComKeyBasic,AVG6701,...,Relstreak_max_eq100,Reltrail_nonnull,Reltrail_eq100,AVG_maxdiff,AVG_mindiff,CountAVG,AVG_Max,AVG_Min,AVGtail_stable,ภาค
0,CU,จันทบุรี,(ห้ามใช้) กางเกงว่ายน้ำบุรุษ ขนาด...............,ธงชัยพาณิชย์,ร้านอื่นๆ,2115004,2115004111808080,1220300022,21150041118080801220300022,250.0,...,25,25,25,0.0,0.0,25,250.0,250.0,25.0,กลาง
1,CU,จันทบุรี,-ค่าท่อพีวีซี ท่อประปา ขนาด 6 หุน (3/4 นิ้ว) ...,ร้านประจักษ์,ร้านอื่นๆ,3140002,3140002111010101,1220300006,31400021110101011220300006,65.0,...,25,25,25,0.0,0.0,25,65.0,65.0,25.0,กลาง
2,CU,จันทบุรี,กระดาษชำระ ชนิดม้วน 2 ชั้น หยาบ บรรจุห่...,โลตัส โกเฟรช สาขาท่าใหม่,Lotus_gofresh,4210011,4210011121010302,1220300193,42100111210103021220300193,40.0,...,3,0,0,8.0,-7.0,21,44.0,35.0,NaN,กลาง
3,CU,จันทบุรี,กระดาษชำระ ชนิดม้วน 2 ชั้น หยาบ บรรจุห่...,ร้านกิจเจริญ,ร้านอื่นๆ,4210011,4210011121010401,1220300017,42100111210104011220300017,40.0,...,12,0,0,0.0,0.0,12,40.0,40.0,NaN,กลาง
4,CU,จันทบุรี,กระดาษชำระ ชนิดม้วน 2 ชั้น หยาบ บรรจุห่...,ห้าง CJ,CJ EXPRESS,4210011,4210011121010401,1220300249,42100111210104011220300249,NaN,...,2,0,0,0.0,0.0,2,40.0,40.0,NaN,กลาง


In [ ]:
General = dfU.copy()
# General = General.rename('')
# GROUP



General = General.rename(columns={"CODE_7_DIGITS":"CODE7"})
General['COMMODITY_CODE'] = General['COMMODITY_CODE'].astype(str).str.zfill(16)
General['DILLER_CODE'] = General['DILLER_CODE'].astype(str).str.zfill(10)
General["CODE7"] = General["CODE7"].astype(str).str.zfill(7)
General = pd.merge(RM,General,on='CODE7',how = 'left')
General = General.sort_values(by='CODE7').reset_index(drop=True)

General = General.rename(columns={'AVGtail_stable': 'ราคาคงที่กี่เดือน'})

# 3. ลบคอลัมน์ที่ไม่ต้องการออก (Drop)
cols_to_drop = [
    'REL6601', 'REL6602', 'REL6603', 'REL6604', 'REL6605', 'REL6606', 
    'REL6607', 'REL6608', 'REL6609', 'REL6610', 'REL6611', 'REL6612', 
    'REL6701', 'REL6702', 'REL6703', 'REL6704', 'REL6705', 'REL6706', 
    'REL6707', 'REL6708', 'REL6709', 'REL6710', 'REL6711', 'REL6712', 
    'REL6801', 'REL6802', 'REL6803', 'REL6804', 'REL6805', 'REL6806', 
    'REL6807', 'REL6808', 'REL6809', 'REL6810', 'REL6811', 'REL6812', 
    'REL6901', 'Rel_Max', 'Rel_Min', 'Relstreak_max_nonnull', 
    'Relstreak_max_eq100', 'Reltrail_nonnull', 'Reltrail_eq100', 
    'AVG_maxdiff', 'AVG_mindiff', 'CountAVG', 'AVG_Max', 'AVG_Min'
]

# ใช้ errors='ignore' เพื่อป้องกันกรณีบางคอลัมน์ไม่มีอยู่ใน DataFrame จะได้ไม่ error
General = General.drop(columns=cols_to_drop, errors='ignore')

output_file = "Steak_Month_ชุดนอกเขต.xlsx"
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    General.to_excel(writer, index=False, sheet_name='Data')
    
    workbook  = writer.book
    worksheet = writer.sheets['Data']
    
    # Formats
    header_fmt = workbook.add_format({'bold':True, 'bg_color':'#D7E4BC', 'border':1, 'align':'center'})
    line_fmt   = workbook.add_format({'bottom': 2}) # เส้นหนาแบ่งกลุ่ม
    num_fmt    = workbook.add_format({'num_format': '#,##0.00'})

    # 1. เขียน Header ใหม่ด้วย Format
    for col_num, value in enumerate(General.columns):
        worksheet.write(0, col_num, value, header_fmt)
    
    # 2. Freeze Panes: แถวที่ 1 (Header), คอลัมน์ที่ 0
    worksheet.freeze_panes(1, 10)
    
    # 3. วาดเส้นแบ่งกลุ่มตาม CODE7
    code7_idx = General.columns.get_loc('CODE7')
    for row in range(1, len(General)):
        if General.iloc[row-1, code7_idx] != General.iloc[row, code7_idx]:
            worksheet.set_row(row, None, line_fmt) # ขีดเส้นใต้แถวก่อนหน้า
            
    # ปรับความกว้างคอลัมน์พื้นฐาน
    worksheet.set_column(0, len(General.columns)-1, 12)

print(f"เสร็จสมบูรณ์! ไฟล์ถูกบันทึกที่: {output_file}")

KeyError: 'PROVINCE_NAME'